<a href="https://colab.research.google.com/github/pranjalgahlot123/toxic-comment-detection/blob/main/toxic_comments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import nltk
from nltk import WordNetLemmatizer, pos_tag, word_tokenize
from nltk.corpus import stopwords as nltk_stopwords
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import pickle

# Data Loading
data = pd.read_csv("/content/drive/MyDrive/toxic comment/FinalBalancedDataset.csv")
data = data.drop("Unnamed: 0", axis=1)

# Text Preprocessing
nltk.download('punkt')
nltk.download('omw-1.4')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

wordnet_lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def prepare_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\']', ' ', text)
    text = word_tokenize(text)
    text = pos_tag(text)
    lemma = [wordnet_lemmatizer.lemmatize(i[0], pos=get_wordnet_pos(i[1])) for i in text]
    lemma = ' '.join(lemma)
    return lemma

# Feature Extraction and Model Training Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=list(nltk_stopwords.words('english')))),
    ('classifier', RandomForestClassifier())
])

# Hyperparameter Tuning with Grid Search (Parallel Processing)
param_grid = {
    'tfidf__max_features': [1000, 5000, 10000],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 10, 20],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(data['tweet'], data['Toxicity'])

# Display the best parameters and ROC AUC
print("Best Parameters:", grid_search.best_params_)
print("Best ROC AUC:", grid_search.best_score_)

# Save the Best Model
best_model = grid_search.best_estimator_
pickle.dump(best_model, open("best_toxicity_model.pkt", "wb"))

# Cross-validation to get a more robust estimate of the model's performance
cv_scores = cross_val_score(best_model, data['tweet'], data['Toxicity'], cv=5, scoring='roc_auc', n_jobs=-1)
print("Cross-Validation Scores:", cv_scores)
print("Mean ROC AUC:", np.mean(cv_scores))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Best Parameters: {'classifier__max_depth': 20, 'classifier__n_estimators': 200, 'tfidf__max_features': 10000}
Best ROC AUC: 0.9359923020579215
Cross-Validation Scores: [0.80503018 0.98225391 0.99022836 0.99484211 0.8569524 ]
Mean ROC AUC: 0.9258613898180746
